<a href="https://colab.research.google.com/github/tavishgrade/nlp-language-model-portfolio/blob/main/Talktorial_4_Embeddings_TavishGrade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Talktorial 4 — Representations & Embeddings: Meaning as Vectors

**Graduate NLP · Week 6 · Exposure #2** (lecture → **talktorial** → peer discussion)

In the lecture, a word became a **vector**: 1st-order counts → 2nd-order → SVD → **word2vec** → contextual.
Here you'll use real **pretrained embeddings**, measure meaning with **cosine**, probe structure with
**analogies** (and *verify* they hold), find out what **one** vector does with the **two** senses of our
running word **bass**, and finally **train your own** word2vec so you feel "learn by prediction" first-hand.

> ⚠️ **Colab users — do this first:** **File → Save a copy in Drive.** Part 1 downloads ~128 MB of
> vectors (cached after the first load); Part 6 trains a small model (~1 min).

**By the end you will be able to:**
1. Load and query **pretrained** word embeddings (GloVe)
2. Rank word similarity with **cosine** — and connect it to Firth's "company it keeps"
3. Test **analogies** (king − man + woman ≈ queen) as claims you **verify**, not trust
4. Show what a **static** vector does with a polysemous word (**bass**) — and find the sense it hides
5. **Train** a word2vec model and see that the **corpus** shapes the meaning

**Submit:** answer the prediction questions below, run all cells (outputs saved), complete the
**AI Verification Log**, answer the **reflection questions** at the end, then upload the
`.ipynb` to Canvas by **Sun 11:59 PM**.

## Before you code: make some predictions

Type your answers in this cell (double-click to edit). **One or two sentences each, and every answer
needs a reason.** Predictions are graded for being real, not for being right.

- **Q1.** Predict the **top-3** words most similar to **"guitar"**. Is *guitar* closer to *drum* or to
  *boat* — and why?
  > *Your answer:* I would predict just based on how I use or see the word: music, play, and drum. Based on context clues, guitar is definitley closer to drum because they would be more commonly seen near each other in sentences.
- **Q2.** Will **king − man + woman** return **queen**? Name **one analogy you expect to FAIL**, and
  say why that one.
  > *Your answer:* I think because the only differences between king and queen are gender, this vector calculation could certainly work. One that might fail is a biased analogy like doctor and the two genders man and woman.
- **Q3.** The word **"bass"** has two senses (a fish; a low musical range). When you ask for its
  nearest neighbors, do you predict **fish** words, **music** words, or a **mix** — and why?
  *(This is the prediction you wrote down during the lecture.)*
  > *Your answer:* If there are two senses, I would assume both prediction types would appear as a mixture, mostly split by occurance count of each type.

## Part 0 — Setup

Provided. Installs gensim if needed and imports what we use (safe to re-run).

> **Colab:** if the install finishes with a message asking you to **restart the runtime**, do that
> (**Runtime → Restart session**) and run this cell again. That is normal, not an error.

In [ ]:
import json, os   # used by the artifact cell in Part 8
import importlib.util, subprocess, sys
if importlib.util.find_spec("gensim") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gensim"])

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec
import numpy as np
print("gensim", gensim.__version__, "ready")

gensim 4.4.0 ready


## Part 1 — Load pretrained embeddings

Provided. `glove-wiki-gigaword-100` = 400k words, each a **100-dimensional** vector, trained on
Wikipedia + Gigaword. First load downloads ~128 MB; after that it's cached.

In [ ]:
model = api.load("glove-wiki-gigaword-100")   # KeyedVectors: word -> 100-d vector
print("vocab size:", len(model.index_to_key))
print("vector for 'bass' has shape:", model["bass"].shape)

[==================================================] 100.0% 128.1/128.1MB downloaded
vocab size: 400000
vector for 'bass' has shape: (100,)


## Part 2 — Cosine similarity = shared company

In lecture, **cosine** measured how much *company* two words keep. Check it on our running example
(music vs. water). `model.similarity(a, b)` returns cosine in roughly 0–1 for related words.

In [ ]:
# TODO: For each pair below, print the cosine similarity with model.similarity(a, b).
#   pairs = [("guitar","drum"), ("guitar","boat"), ("boat","canoe"),
#            ("bass","guitar"), ("bass","fish")]
# Then answer the question in the cell below.

# write out the pairs
pairs = [("guitar","drum"), ("guitar","boat"), ("boat","canoe"),
         ("bass","guitar"), ("bass","fish")]

# iterate through each pair, calculate similarity and print
for a,b in pairs:
    sim = model.similarity(a,b)
    print(f"{a:>10} vs {b:<10} similarity = {sim:.4f}")


    guitar vs drum       similarity = 0.7564
    guitar vs boat       similarity = 0.1958
      boat vs canoe      similarity = 0.5037
      bass vs guitar     similarity = 0.8495
      bass vs fish       similarity = 0.4418


**Which pair is most similar, and which is least?** Does it match the lecture's music-vs-water story — and where does **bass** sit?
> *Your answer:* Well it looks like bass-guitar is the most similar, closest to 1. If I'm looking at the two bass pairs it certainly looks like bass as a music word is more prevalent, but a .4418 similarity of bass vs fish is still significant, landing right near the middle, compared to guitar and boat being very low and .1958. The high, medium and low numbers are expected based on the lecture about the music-water subjects.

## Part 3 — Nearest neighbors

`model.most_similar(word)` returns the closest words by cosine. This is the distributional
hypothesis in action: neighbors are words that keep similar company.

In [ ]:
# TODO: For "guitar" and "river", print the top-10 most_similar words with scores.
# Then answer the question in the cell below.

# iterate through both words
for w in ["guitar", "river"]:
    print(f"\nTop-10 most similar words to '{w}'")
    # calculate the most similar words with library function
    results = model.most_similar(w, topn=10)
    # print out each top ten words in a list
    for word, score in results:
      print(f"   {word:15s}   {score:.4f}")



Top-10 most similar words to 'guitar'
   bass              0.8495
   guitars           0.8201
   vocals            0.8172
   drums             0.8045
   harmonica         0.7939
   percussion        0.7910
   piano             0.7777
   acoustic          0.7759
   saxophone         0.7609
   keyboards         0.7568

Top-10 most similar words to 'river'
   rivers            0.7960
   creek             0.7458
   lake              0.7426
   valley            0.7132
   danube            0.7043
   basin             0.6949
   tributary         0.6928
   canal             0.6885
   flows             0.6843
   estuary           0.6772


**Are these the "company" you expected?** Compare guitar's top 3 with your **Q1** prediction.
> *Your answer:* I got the idea across, and I almost guess singing which is similar to vocals. I definitely didn't guess the plural of guitar which I find a bit funny. It's times like these when I wish I could see the list of words and libraries myself so I have a clear understanding of what I'm working with here.

## Part 4 — Analogies, as a claim you VERIFY

The famous demo: **king − man + woman ≈ queen**. In gensim that is
`most_similar(positive=["woman","king"], negative=["man"])`.

This is exactly an **AI Verification Log** moment: the claim is *"embeddings do analogy arithmetic."*
Don't take it on faith — **test it**, and deliberately find a case where it **breaks**. The failures
are the lesson.

In [ ]:
# TODO: Write a helper analogy(a,b,c) that returns model.most_similar(
#          positive=[b, c], negative=[a], topn=1)   # "a is to b as c is to ?"
# Verify king - man + woman (man:king :: woman:?). Then try THREE more of your
# own, at least one you expect to fail. Print each result.
# Then answer the question in the cell below.

# create the function to find a word analogy guess
def analogy(a, b, c, topn=1):
    # describing analogy 'a is to b' as 'c is to ?'
    return model.most_similar(positive=[b, c], negative=[a], topn=topn)

# creating a list of test lists for analogy testing
tests = [
    ("man", "king", "woman"),
    ("doctor", "woman", "nurse"),
    ("run", "running", "walk"),
    ("up", "down", "left"),
]

# use the function with the test groups
for a, b, c, in tests:
    result = analogy(a, b, c)[0]
    print(f"{a}:{b} :: {c}:  {result[0]}   (score={result[1]:.4f})")

man:king :: woman:  queen   (score=0.7699)
doctor:woman :: nurse:  girl   (score=0.7151)
run:running :: walk:  walking   (score=0.8240)
up:down :: left:  back   (score=0.7452)


**Which analogies held, which failed, and was your Q2 prediction right?** Give your best guess at *why* the failing one fails.
> *Your answer:* For failure, based on what I would guess in these analogies, the first and third held, while the second and last failed. I would guess the second one (doctor-woman, nurse-girl) failed because of gender bias reasons. The last one might have failed because of the numerous amount of options for these four words.

### ✅ Verification checkpoint #1
Ask your AI assistant: *"Do word embeddings reliably solve analogies like king − man + woman = queen?"*
The honest answer is more qualified than the famous demo suggests. **Verify** against your own Part 4
output (including the analogy you expected to fail), the **gensim `most_similar` docs**, and
**SLP3 §Semantic properties of embeddings**. Record it in Part 7.

## Part 5 — The catch: one vector for "bass"

In lecture, **bass** was given two jobs: predict music words **and** predict water words. It still gets
**one** vector. You predicted what that vector looks like (**Q3**). Look.

In [ ]:
# TODO: Print the top-15 most_similar words for "bass", with scores.
# Then answer the question in the cell below.

# calculate the top 15 similar words for 'bass'
bass_top15 = model.most_similar("bass", topn=15)

# display results
print("Top-15 most similar words to 'bass':")
for word, score in bass_top15:
    print(f"   {word:15s}   {score:.4f}")


Top-15 most similar words to 'bass':
   guitar            0.8495
   drums             0.7608
   vocals            0.7466
   saxophone         0.7419
   percussion        0.7332
   guitars           0.7095
   keyboards         0.7055
   trumpet           0.6901
   drum              0.6886
   flute             0.6823
   piano             0.6743
   trombone          0.6738
   harmonica         0.6734
   clarinet          0.6700
   drummer           0.6656


**Fish words, music words, or a mix — and was your Q3 prediction right?** Count them.
> *Your answer:* Hm, for how close bass and guitar were I guess I'm not surprised that the fish sense isn't showing in the top 15 words here. How interesting, I for sure thought it was a mixture!

### Now look closer

If the fish sense were simply *gone*, **bass** would be no closer to *trout* than *guitar* is. Check —
and then see whether the fish sense can be pulled back out.

In [ ]:
# TODO:
#  1. For each of "bass", "guitar", "piano": print its cosine similarity with "trout" and with "fishing".
#  2. Get bass's top-200 neighbours and print the RANK of the first fish word you find
#     (trout, fish, smallmouth, largemouth, walleye, catfish, perch, fishing ...).
#  3. Print the top-6 results of  bass + fishing - guitar :
#        model.most_similar(positive=["bass", "fishing"], negative=["guitar"], topn=6)
# Then answer the questions in the cell below.

# store words for testing
targets = ["trout", "fishing"]
words = ["bass", "guitar", "piano"]

# iterate through test words and calculate cosine similarities
print("Cosine similarities:\n")
for w in words:
    for t in targets:
        sim = model.similarity(w, t)
        print(f"{w:10s} vs {t:10s}  similarity = {sim:.4f}")
    print()

# store test words in list
fish_words = {"trout", "fish", "smallmouth", "largemouth", "walleye",
              "catfish", "perch", "fishing"}

# find top 200 most similar words for bass
neighbors = model.most_similar("bass", topn=200)

# find the ranks of our test words in the neighbors list
rank = None
for i, (word, score) in enumerate(neighbors, start=1):
    if word in fish_words:
        rank = (i, word, score)
        break

# print the top ranked fish related word
print(f"First fish-related neighbor appears at rank {rank[0]}: {rank[1]} (score={rank[2]:.4f})")

# Print the top-6 results of bass + fishing - guitar
print("\nAnalogy: bass + fishing - guitar")
results = model.most_similar(positive=["bass", "fishing"], negative=["guitar"], topn=6)
for word, score in results:
    print(f"  {word:15s}  {score:.4f}")

Cosine similarities:

bass       vs trout       similarity = 0.4934
bass       vs fishing     similarity = 0.4036

guitar     vs trout       similarity = 0.1692
guitar     vs fishing     similarity = 0.2235

piano      vs trout       similarity = 0.1622
piano      vs fishing     similarity = 0.1639

First fish-related neighbor appears at rank 19: smallmouth (score=0.6343)

Analogy: bass + fishing - guitar
  fishermen        0.7345
  trout            0.7079
  trawlers         0.7058
  fish             0.6925
  boats            0.6809
  fishery          0.6616


**Is the fish sense in the vector, or not?** Give your evidence from all three results.
> *Your answer:* There's definitely a fish sense in the vector for bass. When finding the analogy of bass and fishing, guitar and ? we see mostly fish words. By the previous top 15 I would have assume a least some music sense to sneak in there.

**So is "one static vector *averages* the two senses" the right description?** If not, write a better one — and say what decides which sense ends up on top.
> *Your answer:* An average would be too straight-forward of a description for that. Perhaps a more accurate description would use words from the lecture like a weighted mixture of the senses of a word.

### ✅ Verification checkpoint #2
Ask: *"What does a static embedding like word2vec or GloVe do with a word that has two senses, such as
bass — and what would its nearest neighbours look like?"* Verify the answer against **your own Part 5
output** first, then the **Embeddings lecture** (the static → contextual section) and the **SLP3
chapter on vector semantics and embeddings**. Record it in Part 7. You will compute contextual vectors
for *bass* yourselves in the Transformers week.

## Part 6 — Train your own word2vec (learn by prediction)

Pretrained vectors are trained on Wikipedia. Now **you** train one — on 19th-century literature
(NLTK's `gutenberg`: Austen, the KJV Bible, Melville, ...). Same algorithm, *very* different corpus.
Watch how the **corpus makes the meaning** — the lecture's "sugar" point, and its closing question.
The arguments below are the **knobs** from the lecture: vector size, window, min_count, epochs.

In [ ]:
# TODO:
#  1. Build `sents` = lower-cased tokenized sentences from nltk gutenberg.
#  2. Train `my = Word2Vec(sents, vector_size=100, window=5, min_count=5,
#         workers=4, epochs=10)`.
#  3. Print the top-8 neighbors of "king" from BOTH your model (my.wv) and the
#     pretrained `model`. Then answer the question in the cell below.
import nltk
for pkg in ("gutenberg", "punkt_tab", "punkt"):      # provided: gutenberg.sents() needs the sentence tokenizer too
    nltk.download(pkg, quiet=True)
from nltk.corpus import gutenberg

# build sents list, lower-cased tokenized sentences from nltk gutenberg
sents = [[w.lower() for w in sent] for sent in gutenberg.sents()]

# train the model
my = Word2Vec(
    sents,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    epochs=10
)

# compare neighbors of "king" and print out the results
print("My model (Gutenberg) neighbors for 'king':")
for w, score in my.wv.most_similar("king", topn=8):
    print(f"  {w:12s} {score:.4f}")

print("\nPretrained Wikipedia model neighbors for 'king':")
for w, score in model.most_similar("king", topn=8):
    print(f"  {w:12s} {score:.4f}")

My model (Gutenberg) neighbors for 'king':
  governor     0.6874
  david        0.6045
  solomon      0.5681
  jacob        0.5454
  ruler        0.5343
  seer         0.5302
  prince       0.5302
  pharaoh      0.5272

Pretrained Wikipedia model neighbors for 'king':
  prince       0.7682
  queen        0.7508
  son          0.7021
  brother      0.6986
  monarch      0.6978
  throne       0.6920
  kingdom      0.6811
  father       0.6802


**Did it run?** Training takes about a minute on Colab. If it did not finish, what was the error?
> *Your answer:* It finished!

**Why are the two neighbour lists for "king" so different?** Explain it from the two **corpora** — what is *king* keeping company with in each?
> *Your answer:* In the classical biblical corpus of my model, you clearly see the influence of biblical stories and names. WIht the wikipedia model, I imagine the corpus covers a larger swath of king, and in history books that would be rulers.

## Part 7 — AI Verification Log

One block per checkpoint: the **question you asked**, the **claim** that came back, the **source** you
checked it against, your **verdict**, and **what you found when you checked** — what the source actually
said, or what your own output actually showed. Then add at least one claim you chose yourself.

**How the log is graded** — the same four rows as the assignment page:
- **The question.** A real question, and a follow-up where the first answer was thin. *"Isn't cosine
  just the dot product?"* is fishing for agreement; *"What does cosine similarity compute?"* is a question.
- **The source.** The right kind for the kind of claim: the textbook or lecture for a definition, the
  gensim docs for how a function behaves, **your own output** for a claim about what the code does.
- **Thoroughness.** Every checkpoint, plus a claim you chose yourself.
- **What you found.** The finding says what the source *said* or what your output *showed*. Naming a
  caveat, a boundary condition, or a case where the claim breaks is what makes a block excellent
  rather than satisfactory. The AI does not have to be wrong.

*(Double-click this cell to edit. Type after each bold label, then Shift+Enter to render it.)*

**A worked example, filled in so you can see the bar.** This one is not yours to answer:

- **What you asked:** "Does gensim's `most_similar` rank results by cosine similarity?"
- **What it told you:** Yes, it returns the words with the highest cosine similarity to the query.
- **Source checked:** gensim `KeyedVectors` docs, plus my own Part 3 output
- **Verdict:** ⚠️ partly right
- **What you found when you checked:** It ranks by cosine, but on length-normalised vectors, and it
  *excludes the query words* from the results — which is why "king" never appears in its own
  neighbour list, and why an analogy can never return one of its own inputs. The AI did not mention
  the exclusion.

**The same block, written the way that does not count.** Same question, same source:

- **What you found when you checked:** The docs confirm that most_similar uses cosine similarity. ✅

That repeats the claim. It does not say what the docs said, and it did not look at the output. A
finding that could have been written without opening the source is not a check.

---

### Claim 1 — checkpoint #1
- **What you asked:** Do word embeddings reliably solve analogies like king − man + woman = queen?
- **What it told you:** No, they only sometimes solve those analogies and not reliably in any way. The famous demo works because it just happens that the vectors align in a strong linear semantic direction in the embedding space.
- **Source checked:** Lecture, SLP3, my results
- **Verdict:** ⚠️ partly right
- **What you found when you checked:** My results from the homework and the AI answer certainly align with the conclusion that not all analogies are solved with word embeddings. However, the lecture and the book give more details as to reasons why that may be, such as common ones like the relations not being well-represented in the corpus or the relations aren't so linear like my example of up-down left-right. The book goes even further with vocabulary facts like how embeddings capture distributional similarity, not true semantics.

### Claim 2 — checkpoint #2
- **What you asked:** What does a static embedding like word2vec or GloVe do with a word that has two senses, such as
bass — and what would its nearest neighbours look like?
- **What it told you:** A static embedding assigns one signle vector to bass that ends up being a mixture of all the contexts in which the word appears in the training corpus.
- **Source checked:** My own results, lecture, SLP3
- **Verdict:** ⚠️ partly right
- **What you found when you checked:** Both the AI and my work show that dual senses of a word whittle down to one mathematical representation in the programming. However, describing it as a simple mixture misses a lot of the nuance that the book and lecture cover on the topic. We get more details such as a fix for this is transformers/contextual embeddings that produce different vectors for different uses.

### Claim 3 — one you chose yourself
*A claim about what the vectors do is checked against your own output — an analogy from Part 4, or
the bass numbers from Part 5, belong here.*
- **What you asked:** Does the vector for bass contain the fish sense?
- **What it told you:** The AI said that yes it contains it because bass is similar to trout and fishing.
- **Source checked:** My output
- **Verdict:** ⚠️ partly right
- **What you found when you checked:** Of course the AI got the general idea across corrent, but the details of my output show be the lengths at which we can verify this. In particular, when the find the analogy of bass and fishing - guitar, the results are fish words, showing us how aligned bass is with the fish sense.

**Verdict key:** ✅ correct · ⚠️ partly right / needs caveat · ❌ wrong.

## Part 8 — Save your artifact (shared format → final project)

In [ ]:
# TODO: Build a dict `artifact` in the shared envelope
#   {"week": 6, "task": "embeddings", "source": "...", "outputs": {...}}
# `outputs` is a DICT with named keys. Fill it from the variables and calls in this
# notebook, not by typing values in: cosine(guitar,drum), cosine(guitar,boat), the king
# analogy result, bass's top-5 neighbors, cosine(bass,trout), cosine(guitar,trout), and
# your trained model's top-5 neighbors of "king". Wrap numpy numbers in float(...) or
# json.dump will refuse them. Save it with json.dump as talktorial4_artifact.json, then print it.
artifact = {
    "week": 6,
    "task": "embeddings",
    "source": "gutenberg + glove-wiki-gigaword-100",
    "outputs": {
        # Part 4 cosine similarities
        "cosine_guitar_drum": float(model.similarity("guitar", "drum")),
        "cosine_guitar_boat": float(model.similarity("guitar", "boat")),

        # King analogy result (man:king :: woman:?)
        "king_analogy": analogy("man", "king", "woman")[0][0],

        # Bass top-5 neighbors (from pretrained model)
        "bass_top5": [
            (w, float(score))
            for w, score in model.most_similar("bass", topn=5)
        ],

        # Part 5 cosine similarities
        "cosine_bass_trout": float(model.similarity("bass", "trout")),
        "cosine_guitar_trout": float(model.similarity("guitar", "trout")),

        # Gutenberg-trained model's top-5 neighbors of "king"
        "my_king_top5": [
            (w, float(score))
            for w, score in my.wv.most_similar("king", topn=5)
        ]
    }
}

# Save it
with open("talktorial4_artifact.json", "w") as f:
    json.dump(artifact, f, indent=2)

# Print it
print(json.dumps(artifact, indent=2))


{
  "week": 6,
  "task": "embeddings",
  "source": "gutenberg + glove-wiki-gigaword-100",
  "outputs": {
    "cosine_guitar_drum": 0.7564401626586914,
    "cosine_guitar_boat": 0.19577935338020325,
    "king_analogy": "queen",
    "bass_top5": [
      [
        "guitar",
        0.8494914770126343
      ],
      [
        "drums",
        0.7608356475830078
      ],
      [
        "vocals",
        0.7466387748718262
      ],
      [
        "saxophone",
        0.7419118285179138
      ],
      [
        "percussion",
        0.7332254648208618
      ]
    ],
    "cosine_bass_trout": 0.49343112111091614,
    "cosine_guitar_trout": 0.16922181844711304,
    "my_king_top5": [
      [
        "governor",
        0.6874030232429504
      ],
      [
        "david",
        0.604533314704895
      ],
      [
        "solomon",
        0.5680971145629883
      ],
      [
        "jacob",
        0.5453578233718872
      ],
      [
        "ruler",
        0.534338653087616
      ]
    ]
  }

In [ ]:
# Provided: checks the artifact is in the shared shape. If it fails, fix the cell above.
import json, os
assert os.path.exists("talktorial4_artifact.json"), "talktorial4_artifact.json was not written"
_a = json.load(open("talktorial4_artifact.json"))
_missing = {"week", "task", "source", "outputs"} - set(_a)
assert not _missing, f"envelope is missing {_missing}"
assert isinstance(_a["outputs"], dict) and _a["outputs"], "outputs must be a non-empty dict with named keys"
print("artifact OK — outputs keys:", sorted(_a["outputs"]))

artifact OK — outputs keys: ['bass_top5', 'cosine_bass_trout', 'cosine_guitar_boat', 'cosine_guitar_drum', 'cosine_guitar_trout', 'king_analogy', 'my_king_top5']


## Before you submit — a few questions, honestly answered

These are not graded for being impressive. They are graded for being real, and they are
how I find out what is not landing while there is still time to do something about it.
*(Double-click this cell to type your answers, then Shift+Enter.)*

**1. What surprised you?**
*(Scroll back to your predictions at the top — where was your intuition wrong, and why?)*

> *Your answer:* It was surprising to see how prominently the music sense of bass showed in the top word associations. Based on our discussions, I would have thought there would be a bit more mixture at the top. Of course, the mix did come back in the end.

**2. What got you stuck this week, and what did you do about it?**
*("I am still stuck on X" is a complete and fully-credited answer.)*

> *Your answer:* The analogies in general were hard to understand for me, in particular how the math of it all works in the backend.

**3. What is one question you still have?**

> *Your answer:* I'm still curious how this looks in the next step with chat models, and how they are so effective at figuring out which sense of words is used.

---

### Working with AI

*There is no right answer here — "it frustrated me" and "I am not sure it helped" are as useful to me as anything positive. Answer honestly rather than helpfully.*

**a. Which AI did you use this week?** Name the assistant, and the model if you know it — e.g. "ChatGPT", "Claude", "Gemini", "Copilot", a local model, or "none this week", which is a perfectly good answer.

> *Your answer:* Copilot

**b. After completing this week’s AI-supported task and checking the AI’s response, what did you think and feel about your experience with AI?**

*You might reflect on things such as what surprised you, what confused or frustrated you, what helped you learn, what made you feel more or less confident, or whether checking the AI’s response changed your thinking or feelings.*

> *Your answer:* I can see that it covers details for a lot of things, but can sometimes miss the variants of options that the book covers.



## Wrap-up

You measured meaning with **cosine** (Firth's "company," as arithmetic), probed structure with
**analogies** — and, crucially, **verified** them instead of trusting them. You found what one static
vector does with **bass**: one sense on the surface, the other underneath — the failure that motivates
the **contextual** embeddings (ELMo, BERT) from the end of the lecture, which you will compute
yourselves in the Transformers week. And by **training your own** word2vec, you felt "learn by
prediction" and saw the corpus decide the meaning.

**Optional extensions:** try analogies in your own domain; compute the cosine of two *sentences* by
averaging their word vectors (a preview of sentence embeddings / RAG); retrain Part 6 with
`sg=1` (skip-gram) vs `sg=0` (CBOW) and compare.

**Submit:** Run all cells, then **File → Download .ipynb** → upload to Canvas with the Verification
Log completed.